# Análise - Pontuação Média dos Campeões por Ano

Nosso objetivo nesta análise é avaliar a pontuação média dos campeões de pilotos por ano, para entender:

- Se os campeões estão se distanciando mais do segundo colocado ao longo do tempo;
- Se há épocas de maior equilibrio ou de domínio extremo.



Para isso vamos realizar o tratamento das base

In [9]:
import pandas as pd

colocacao = pd.read_csv("driver_standings.csv", delimiter=",")
corridas = pd.read_csv("races.csv", delimiter=",")
pilotos = pd.read_csv("drivers.csv", delimiter=",")

# Juntando a base standings com a base corridas para pegar os anos
colocacao = colocacao.merge(corridas[["raceId", "year"]], on="raceId")

# Pegando a colocação após a última corrida de cada ano
ultima_corrida_id = corridas.groupby("year")["raceId"].max().reset_index()
colocacao_final = colocacao.merge(ultima_corrida_id, on=["year", "raceId"])

campeoes = colocacao_final[colocacao_final["position"] == 1]

campeoes = campeoes.merge(pilotos[["driverId", "forename", "surname"]], on="driverId")
campeoes["driver_name"] = campeoes["forename"] + ' ' + campeoes["surname"]

Agora é hora de analisar 

In [10]:
pontuacao_campeos = campeoes[["year", "points", "driver_name"]]

print(pontuacao_campeos)

    year  points         driver_name
0   2008    98.0      Lewis Hamilton
1   2007   110.0      Kimi Räikkönen
2   2006   134.0     Fernando Alonso
3   2005   133.0     Fernando Alonso
4   2004   148.0  Michael Schumacher
..   ...     ...                 ...
70  2020   347.0      Lewis Hamilton
71  2021   395.5      Max Verstappen
72  2022   454.0      Max Verstappen
73  2023   575.0      Max Verstappen
74  2024   437.0      Max Verstappen

[75 rows x 3 columns]


In [12]:
# Pegando os vices (posição 2 na última corrida de cada ano)
vices = colocacao_final[colocacao_final["position"] == 2]

# Juntando com os nomes dos pilotos vices
vices = vices.merge(pilotos[["driverId", "forename", "surname"]], on="driverId")
vices["vice_name"] = vices["forename"] + ' ' + vices["surname"]

# Juntando com os campeões por ano
dominancia = campeoes[["year", "points", "driver_name"]].merge(
    vices[["year", "points", "vice_name"]],
    on="year",
    suffixes=("_campeao", "_vice")
)

# Diferença de pontuação entre campeão e vice
dominancia["diferenca"] = dominancia["points_campeao"] - dominancia["points_vice"]

print(dominancia)

    year  points_campeao         driver_name  points_vice           vice_name  \
0   2008            98.0      Lewis Hamilton         97.0        Felipe Massa   
1   2007           110.0      Kimi Räikkönen        109.0      Lewis Hamilton   
2   2006           134.0     Fernando Alonso        121.0  Michael Schumacher   
3   2005           133.0     Fernando Alonso        112.0      Kimi Räikkönen   
4   2004           148.0  Michael Schumacher        114.0  Rubens Barrichello   
..   ...             ...                 ...          ...                 ...   
70  2020           347.0      Lewis Hamilton        223.0     Valtteri Bottas   
71  2021           395.5      Max Verstappen        387.5      Lewis Hamilton   
72  2022           454.0      Max Verstappen        308.0     Charles Leclerc   
73  2023           575.0      Max Verstappen        285.0        Sergio Pérez   
74  2024           437.0      Max Verstappen        374.0        Lando Norris   

    diferenca  
0         1